# Snowflake — Target Tables & Verification Queries

All scripts run on Snowflake.

| Cell | Purpose | When to Run |
|------|---------|-------------|
| 1 | Create target tables (SCD2 + SCD1) | Before demo (one-time setup) |
| 2 | Verify SCD2 history — single customer | After incremental migration |
| 3 | Verify SCD2 summary — row counts | After incremental migration |
| 4 | Verify SCD2 Products — price history | After incremental migration |
| 5 | Verify SCD1 Orders — total count | After incremental migration |

> See `DEMO_PLAN.md` for the full recording guide.
> See `mssql.ipynb` for MSSQL source data scripts.

## 1. Create Target Tables (SCD2 + SCD1)
Creates target tables with SCD2 meta columns (`IS_CURRENT`, `EFF_START_DATE`, `EFF_END_DATE`).

In [ ]:
%%sql -r create_customers
USE WAREHOUSE COMPUTE_WH;
USE DATABASE ANALYTICS;
CREATE SCHEMA IF NOT EXISTS PUBLIC;
USE SCHEMA PUBLIC;

-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  SCD2 TABLE: CUSTOMERS                                                      ║
-- ║  IS_CURRENT / EFF_START_DATE / EFF_END_DATE enable history tracking          ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

CREATE OR REPLACE TABLE ANALYTICS.PUBLIC.CUSTOMERS (
    CUSTOMERID      INT,
    FIRSTNAME       VARCHAR(50),
    LASTNAME        VARCHAR(50),
    EMAIL           VARCHAR(100),
    PHONE           VARCHAR(20),
    CITY            VARCHAR(50),
    STATE           VARCHAR(30),
    COUNTRY         VARCHAR(30),
    ZIPCODE         VARCHAR(10),
    CUSTOMERTYPE    VARCHAR(20),
    CREDITLIMIT     DECIMAL(12,2),
    ISACTIVE        BOOLEAN,
    CREATEDAT       TIMESTAMP_NTZ,
    UPDATEDAT       TIMESTAMP_NTZ,
    -- SCD2 columns
    IS_CURRENT      BOOLEAN DEFAULT TRUE,
    EFF_START_DATE  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    EFF_END_DATE    TIMESTAMP_NTZ
);

In [ ]:
%%sql -r create_products
-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  SCD2 TABLE: PRODUCTS                                                       ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

CREATE OR REPLACE TABLE ANALYTICS.PUBLIC.PRODUCTS (
    PRODUCTID       INT,
    PRODUCTNAME     VARCHAR(100),
    CATEGORY        VARCHAR(50),
    SUBCATEGORY     VARCHAR(50),
    BRAND           VARCHAR(50),
    SKU             VARCHAR(30),
    UNITPRICE       DECIMAL(10,2),
    COSTPRICE       DECIMAL(10,2),
    WEIGHT          DECIMAL(8,2),
    ISACTIVE        BOOLEAN,
    CREATEDAT       TIMESTAMP_NTZ,
    UPDATEDAT       TIMESTAMP_NTZ,
    -- SCD2 columns
    IS_CURRENT      BOOLEAN DEFAULT TRUE,
    EFF_START_DATE  TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    EFF_END_DATE    TIMESTAMP_NTZ
);

In [ ]:
%%sql -r create_orders
-- ╔══════════════════════════════════════════════════════════════════════════════╗
-- ║  SCD1 TABLE: ORDERS (no history, simple upsert)                             ║
-- ╚══════════════════════════════════════════════════════════════════════════════╝

CREATE OR REPLACE TABLE ANALYTICS.PUBLIC.ORDERS (
    ORDERID         INT,
    CUSTOMERID      INT,
    PRODUCTID       INT,
    ORDERDATE       DATE,
    SHIPDATE        DATE,
    QUANTITY        INT,
    UNITPRICE       DECIMAL(10,2),
    DISCOUNT        DECIMAL(5,2),
    TOTALAMOUNT     DECIMAL(12,2),
    ORDERSTATUS     VARCHAR(20),
    PAYMENTMETHOD   VARCHAR(20),
    SHIPPINGMETHOD  VARCHAR(20),
    REGION          VARCHAR(30),
    CREATEDAT       TIMESTAMP_NTZ,
    UPDATEDAT       TIMESTAMP_NTZ
);

In [ ]:
%%sql -r create_wrk
-- WRK staging schema (used by COPY INTO during migration)
CREATE SCHEMA IF NOT EXISTS ANALYTICS.PUBLIC_WRK;

## 2. Verify SCD2 — Customer History
Run after incremental migration to show SCD2 versioning in the demo.

In [ ]:
%%sql -r scd2_customer_history
-- Show SCD2 history for CustomerID=1 (old version vs new version)
SELECT CUSTOMERID, FIRSTNAME, EMAIL, CITY,
       IS_CURRENT, EFF_START_DATE, EFF_END_DATE
FROM ANALYTICS.PUBLIC.CUSTOMERS
WHERE CUSTOMERID = 1
ORDER BY EFF_START_DATE;
-- Expected: 2 rows
--   Row 1: IS_CURRENT=FALSE, EFF_END_DATE set (expired old version)
--   Row 2: IS_CURRENT=TRUE,  EFF_END_DATE=NULL (current version)

## 3. Verify SCD2 — Row Count Summary

In [ ]:
%%sql -r scd2_summary
-- Current vs historical row counts
SELECT
    COUNT(*) AS TOTAL_ROWS,
    SUM(CASE WHEN IS_CURRENT = TRUE THEN 1 ELSE 0 END) AS CURRENT_ROWS,
    SUM(CASE WHEN IS_CURRENT = FALSE THEN 1 ELSE 0 END) AS HISTORICAL_ROWS
FROM ANALYTICS.PUBLIC.CUSTOMERS;
-- Expected after incremental: TOTAL=100,060 | CURRENT=100,010 | HISTORICAL=50

## 4. Verify SCD2 — Products Price History

In [ ]:
%%sql -r scd2_products
-- Products with price history (SCD2 preserves old prices)
SELECT PRODUCTID, PRODUCTNAME, UNITPRICE, IS_CURRENT, EFF_START_DATE, EFF_END_DATE
FROM ANALYTICS.PUBLIC.PRODUCTS
WHERE PRODUCTID IN (1, 2, 3)
ORDER BY PRODUCTID, EFF_START_DATE;
-- Expected: Each product shows OLD price (IS_CURRENT=FALSE) + NEW price (IS_CURRENT=TRUE)

## 5. Verify SCD1 — Orders Count

In [ ]:
%%sql -r scd1_orders
-- SCD1: simple count (no versioning, just upserted)
SELECT COUNT(*) AS TOTAL_ORDERS FROM ANALYTICS.PUBLIC.ORDERS;
-- Expected after incremental: 1,000,100 (original 1M + 100 new)